In [ ]:
!pip install -q pandas

In [ ]:
!pip install torch

In [ ]:
!pip install torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from detanet_model import *
from torch_geometric.loader import DataLoader
import pandas as pd

In [ ]:
'''First, the pytorch library can be used to load the dataset, which consists of 130K molecules, after importing the pytorch library.'''
dataset=torch.load('../data/qm9s.pt', weights_only=False)

len(dataset)

In [ ]:
import pandas as pd
pd.read_csv('../data/qm9s_csv/000003.csv')

In [ ]:
'''where each molecule is a gemetric.data.Data. containing coordinates, atomic numbers, edge indices, and various properties.'''
dataset[0]

In [ ]:
'''We divided the dataset evenly and used 5% of the data for testing and other for training:'''
train_datasets=[]
val_datasets=[]
for i in range(len(dataset)):
    if i%20==0:
        val_datasets.append(dataset[i])
    else:
        train_datasets.append(dataset[i])
        
len(train_datasets),len(val_datasets)

In [ ]:
'''Using torch_Geometric.dataloader.DataLoader Converts a dataset into a batch of 64 molecules of training data.'''
bathes=64
trainloader=DataLoader(train_datasets,batch_size=bathes,shuffle=True)
valloader=DataLoader(val_datasets,batch_size=bathes,shuffle=True)

In [ ]:
'''After loading the dataset, we train a model using NPA charge as an example.
 	Firstly, construct an untrained model:'''
model=DetaNet(num_features=128,
                act='swish',
                maxl=3,
                num_block=3,
                radial_type='trainable_bessel',
                num_radial=32,
                attention_head=8,
                rc=5.0,
                dropout=0.0,
                use_cutoff=False,
                max_atomic_number=9,
                atom_ref=None,
                scale=None,
                scalar_outsize=1,
                irreps_out=None,
                summation=False,
                norm=False,
                out_type='scalar',
                grad_type=None ,
                device=torch.device('cuda'))

model.train()

In [ ]:
'''Next, define the trainer and the parameters used for training.'''
class Trainer:
    def __init__(self,model,train_loader,val_loader=None,loss_function=l2loss,device=torch.device('cuda'),
                 optimizer='Adam_amsgrad',lr=5e-4,weight_decay=0):
        self.opt_type=optimizer
        self.device=device
        self.model=model
        self.train_data=train_loader
        self.val_data=val_loader
        self.device=device
        self.opts={'AdamW':torch.optim.AdamW(self.model.parameters(),lr=lr,amsgrad=False,weight_decay=weight_decay),
              'AdamW_amsgrad':torch.optim.AdamW(self.model.parameters(),lr=lr,amsgrad=True,weight_decay=weight_decay),
              'Adam':torch.optim.Adam(self.model.parameters(),lr=lr,amsgrad=False,weight_decay=weight_decay),
              'Adam_amsgrad':torch.optim.Adam(self.model.parameters(),lr=lr,amsgrad=True,weight_decay=weight_decay),
              'Adadelta':torch.optim.Adadelta(self.model.parameters(),lr=lr,weight_decay=weight_decay),
              'RMSprop':torch.optim.RMSprop(self.model.parameters(),lr=lr,weight_decay=weight_decay),
              'SGD':torch.optim.SGD(self.model.parameters(),lr=lr,weight_decay=weight_decay)
        }
        self.optimizer=self.opts[self.opt_type]
        self.loss_function=loss_function
        self.step=-1
    def train(self,num_train,targ,stop_loss=1e-8, val_per_train=50, print_per_epoch=10):
        self.model.train()
        len_train=len(self.train_data)
        for i in range(num_train):
            val_datas=iter(self.val_data)
            for j,batch in enumerate(self.train_data):
                self.step=self.step+1
                torch.cuda.empty_cache()
                self.optimizer.zero_grad()
                out = self.model(pos=batch.pos.to(self.device), z=batch.z.to(self.device),
                                     batch=batch.batch.to(self.device))
                target = batch[targ].to(self.device)
                loss = self.loss_function(out.reshape(target.shape),target)
                loss.backward()
                self.optimizer.step()
                if (self.step%val_per_train==0) and (self.val_data is not None):
                    val_batch = next(val_datas)
                    val_target=val_batch[targ].to(self.device).reshape(-1)

                    val_out = self.model(pos=val_batch.pos.to(self.device), z=val_batch.z.to(self.device),
                                             batch=val_batch.batch.to(self.device)).reshape(val_target.shape)
                    val_loss = self.loss_function(val_out, val_target).item()
                    val_mae=l1loss(val_out, val_target).item()
                    val_R2=R2(val_out,val_target).item()
                    if self.step % print_per_epoch==0:
                        print('Epoch[{}/{}],loss:{:.8f},val_loss:{:.8f},val_mae:{:.8f},val_R2:{:.8f}'
                              .format(self.step,num_train*len_train,loss.item(),val_loss,val_mae,val_R2))

                    assert (loss > stop_loss) or (val_loss > stop_loss),'Training and prediction Loss is less' \
                                                                        ' than cut-off Loss, so training stops'
                elif (self.step % print_per_epoch == 0) and (self.step%val_per_train!=0):
                    print('Epoch[{}/{}],loss:{:.8f}'.format(self.step,num_train*len_train, loss.item()))
                    
    def load_state_and_optimizer(self,state_path=None,optimizer_path=None):
        if state_path is not None:
            state_dict=torch.load(state_path)
            self.model.load_state_dict(state_dict)
        if optimizer_path is not None:
            self.optimizer=torch.load(optimizer_path)

    def save_param(self,path):
        torch.save(self.model.state_dict(),path)

    def save_model(self,path):
        torch.save(self.model, path)

    def save_opt(self,path):
        torch.save(self.optimizer,path)

    def params(self):
        return self.model.state_dict()
    

In [ ]:
'''Then, modify the data type and device type'''
device=torch.device('cpu')
dtype=torch.float32
model=model.to(dtype)
model=model.to(device)

In [ ]:
'''Finally, using the trainer, training 20 times from a 5e-4 learning rate'''
trainer=Trainer(model,train_loader=trainloader,val_loader=valloader,loss_function=l2loss,lr=5e-4,weight_decay=0,optimizer='AdamW')

In [ ]:
trainer.train(num_train=1,targ='npacharge')

In [ ]:
'''After the training is completed, take out the learnable parameters and save them as a. pth file.'''
state_dict=model.state_dict

state_dict()

In [ ]:
torch.save(model.state_dict(),'trained_param/test_param.pth')

In [ ]:
'''When needed model , simply load parameters on the untrained model. The parameters we have trained are saved in trained_Param/.'''
state_dict=torch.load('trained_param/qm9spectra/npacharge.pth')
model.load_state_dict(state_dict)